# Data checks

What the panel series are, how the bridge-latency draws compare with observed transfers, and a unit check on the Uniswap prices. The numbers come from `crosschain_arb.checks` and are also saved by `scripts/run_data_checks.py`.

In [1]:
import pandas as pd
from IPython.display import Image, display

pd.options.display.float_format = "{:,.4f}".format


def show(path):
    display(Image(filename=str(path)))

In [2]:
import json

from crosschain_arb import checks
from crosschain_arb.config import RESULTS_DIR
from crosschain_arb.io import data_path, load_latency, load_panel

panel = load_panel()
dex = pd.read_parquet(data_path("dex_midprices.parquet"))

## 1. Panel series

In [3]:
pd.Series(checks.panel_checks(panel)).to_frame("value")

,value
rows,1017975
first,2023-03-12 09:38:00.400000
last,2023-06-30 23:59:57.087000
basis_equals_close_minus_rolling50,True
corr_depth_vs_rolling_volume,1.0000
latency_lag1_autocorr,-0.0010
panel_latency_percentiles_s,"{'p50': 89.5, 'p90': 386.3, 'p99': 1270.8}"
share_spike_stored_flag,0.0919
share_spike_recomputed_p95,0.0500


`basis_t` equals the Binance close minus its 50-tick moving average, `depth` tracks a rolling Binance volume (correlation 0.999996) and `latency_t` has no autocorrelation: it is an independent draw for every tick. They are proxies for the Uniswap L2−L1 basis, on-curve depth and bridge latency.

## 2. Latency draws versus observed transfers

In [4]:
half_lives = {
    "proxy_basis_1s": json.loads((RESULTS_DIR / "h1.json").read_text())["half_life_s"],
    "uniswap_basis_1s": json.loads((RESULTS_DIR / "uniswap_basis.json").read_text())["half_life_by_grid_s"][
        "1s"
    ],
}
lat = checks.latency_fit_checks(load_latency(), half_lives)
pd.DataFrame({"observed": lat["empirical_percentiles_s"], "fitted lognormal": lat["fitted_percentiles_s"]})

,observed,fitted lognormal
p50,49.0000,89.5477
p90,646.0000,386.2794
p99,698.8000,"1,271.9392"


In [5]:
print(
    f"n = {lat['n']:,} | lognormal mu = {lat['lognormal_mu']:.3f}, sigma = {lat['lognormal_sigma']:.3f} | KS statistic {lat['ks_statistic']:.2f}"
)
print({k: f"{100 * v:.2f}%" for k, v in lat["share_faster_than_half_life"].items()})

n = 9,362 | lognormal mu = 4.495, sigma = 1.141 | KS statistic 0.27
{'proxy_basis_1s': '0.01%', 'uniswap_basis_1s': '71.96%'}


The maximum-likelihood lognormal recovers the parameters used to simulate `latency_t` (μ = 4.495, σ = 1.141), but it describes the observed transfers only roughly (KS statistic 0.27): its median is 90 s against an observed 49 s, and its 99th percentile 1,272 s against 699 s. The last line is the share of observed transfers that settle faster than each half-life.

## 3. Uniswap price units

In [6]:
pd.Series(checks.unit_checks(panel, dex)).to_frame("value")

,value
dex_first,2023-03-01 00:00:11
dex_last,2023-09-30 23:59:58.005000
corr_binance_vs_price_l1,0.9993
corr_binance_vs_price_l2,0.9994


The raw prices embed the 10¹² decimals factor of the WETH/USDC pair and the Ethereum pool quotes the inverse price; `prices.to_usd` converts both, and the result tracks Binance (correlation ≈ 0.999).